# CXR Essential-Tag Evaluation — Single Pipeline

This notebook goes straight from your institution's **raw CXR DICOM folder** to the **final 5 quality-evaluation CSV files**.
After editing only the path variables in the **① User Configuration cell** below to match your environment,
run the cells **in order from top to bottom**.

## Pipeline Overview
```
DICOM (.dcm) folder
   │  ② dicom_to_parquet.py  (extract metadata tags)
   ▼
hive-partitioned parquet dataset  (modality=.../tag=.../part-*.parquet)
   │  ③ merge into a single parquet
   ▼
single .parquet (long-format)
   │  ④ clean df_input  (column/schema conversion · extract top-level tags · merge multi-valued tags)
   ▼
Evaluator df_input
   │  ⑤ CxrEssentialTagEvaluator evaluation
   ▼
final 5 CSV files
```

## Final 5 CSV files produced (`OUTPUT_DIR`)
1. `cxr_overall_rates.csv` — per-tag completeness / conformance
2. `cxr_conformance_summary.csv` — CS value pass/partial/none summary
3. `cxr_conformance_partial.csv` — details of partial matches (allowable value found as a word)
4. `cxr_conformance_none.csv` — details of complete mismatches
5. `cxr_unconformed_values.csv` — consolidated non-conforming unique values

> Reference code
> - Metadata extraction: <https://github.com/dr-you-group/dicom_to_parquet>
> - Quality evaluation: <https://github.com/dr-you-group/dicomHeterogeneity>

## ① User Configuration Step
**Only the path variables in this cell need to be edited** to match your local environment. (No other cells need changes)

- `pathlib.Path` is used for cross-platform (Windows/Unix) compatibility.
- `MODALITY_FILTER`: modality filter to keep only CXR. An empty list (`[]`) uses all modalities.

In [ ]:
from pathlib import Path

# ===== Paths to edit for your institution (this is the only input needed!) =====
DICOM_ROOT          = Path() # Top-level folder of your institution's CXR DICOM (.dcm) files (recursively searched)
WORK_DIR            = Path() # Folder to clone the two git repos shared earlier and store intermediate outputs

PARQUET_OUT_DIR     = Path() # Path to save the dicom_to_parquet partitioned dataset (saved split across folders)
SINGLE_PARQUET_PATH = Path() # Path for the merged single .parquet file
OUTPUT_DIR          = Path() # Folder for the final 5 CSV files (as requested)

# CXR-only filter (empty list = use all modalities). CR=Computed Radiography, DX=Digital Radiography
MODALITY_FILTER     = ["CR", "DX"]

# Create required folders (creates if missing, leaves as-is if present)
for _p in (WORK_DIR, PARQUET_OUT_DIR, SINGLE_PARQUET_PATH.parent, OUTPUT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print("Configuration complete")
print(f"  DICOM_ROOT          = {DICOM_ROOT}")
print(f"  WORK_DIR            = {WORK_DIR}")
print(f"  PARQUET_OUT_DIR     = {PARQUET_OUT_DIR}")
print(f"  SINGLE_PARQUET_PATH = {SINGLE_PARQUET_PATH}")
print(f"  OUTPUT_DIR          = {OUTPUT_DIR}")
print(f"  MODALITY_FILTER     = {MODALITY_FILTER}")

# Check that DICOM_ROOT exists and has content
if not DICOM_ROOT.exists():
    raise FileNotFoundError(f"[Error] DICOM_ROOT folder does not exist: {DICOM_ROOT}")
_n_dcm = sum(1 for _ in DICOM_ROOT.rglob("*.dcm"))
print(f"\nNumber of .dcm files under DICOM_ROOT: {_n_dcm}")
if _n_dcm == 0:
    raise FileNotFoundError(
        f"[Error] No .dcm files found under DICOM_ROOT: {DICOM_ROOT}\n"
        f"       Check that the path is correct and the extension is .dcm.")

## ② Environment Preparation Step
Check that the required Python packages are installed, and prepare the two reference repos
(`dicom_to_parquet`, `dicomHeterogeneity`) under `WORK_DIR`.

- If you need to install packages, please uncomment the `%pip install` line below and run it.
- If you have already cloned the two shared git repos into `WORK_DIR`, cloning is skipped; otherwise this cell runs `git clone`.

In [ ]:
# Required packages (uncomment below if installation is needed)
# %pip install pydicom pyarrow pandas numpy openpyxl

import sys, subprocess, json

# --- Verify required package imports ---
try:
    import pydicom, pyarrow, pandas as pd, numpy as np, openpyxl  # noqa: F401
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    print("Package imports succeeded")
    print(f"  python  = {sys.version.split()[0]}")
    print(f"  pydicom = {pydicom.__version__}, pyarrow = {pyarrow.__version__}, pandas = {pd.__version__}")
except ImportError as e:
    raise ImportError(
        f"[Error] Required package missing: {e}\n"
        f"       Uncomment '%pip install pydicom pyarrow pandas numpy openpyxl' in the cell above and install.")

# --- Prepare the two repos (git clone if missing, skip if present) ---
REPOS = {
    "dicom_to_parquet":   "https://github.com/dr-you-group/dicom_to_parquet.git",
    "dicomHeterogeneity": "https://github.com/dr-you-group/dicomHeterogeneity.git",
}
for name, url in REPOS.items():
    target = WORK_DIR / name
    if target.exists():
        print(f"[skip] Already exists: {target}")
        continue
    print(f"[clone] {url} -> {target}")
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(target)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f"[Error] git clone failed: {url}\n{r.stderr}\n"
            f"       Check your internet/git installation, or manually copy the repo to {target}.")
    print("       clone complete")

# Verify core files exist
DICOM_TO_PARQUET_PY = WORK_DIR / "dicom_to_parquet" / "dicom_to_parquet.py"
EVALUATOR_DIR       = WORK_DIR / "dicomHeterogeneity" / "DicomStandardEvaluator" / "Evaluator"
REFERENCE_XLSX      = WORK_DIR / "dicomHeterogeneity" / "files" / "CxrEssentialTags" / "CxrEssentialTags_ReferenceSet.xlsx"
for p in (DICOM_TO_PARQUET_PY, EVALUATOR_DIR, REFERENCE_XLSX):
    if not p.exists():
        raise FileNotFoundError(f"[Error] Required file/folder missing: {p}")
print("\nReference repos/files verified")

## ③ Metadata Extraction Step
Runs `dicom_to_parquet.py` via `subprocess` to expand all DICOM header tags into
long-format, producing a **hive-partitioned parquet dataset** in `PARQUET_OUT_DIR`.

- Options: `--dicom_root DICOM_ROOT --out_dir PARQUET_OUT_DIR --skip_pixel_data`
  (`--skip_pixel_data` excludes pixel data for faster processing)
- After running, prints `_build_summary.json` to check the **number of files processed / failed**.

In [ ]:
# Run dicom_to_parquet.py (uses the same python as the current kernel: sys.executable)
cmd = [
    sys.executable, str(DICOM_TO_PARQUET_PY),
    "--dicom_root", str(DICOM_ROOT),
    "--out_dir",    str(PARQUET_OUT_DIR),
    "--skip_pixel_data",
]
print("Command:\n  " + " ".join(cmd) + "\n")
proc = subprocess.run(cmd, capture_output=True, text=True)
print("----- STDOUT -----")
print(proc.stdout[-4000:])
if proc.stderr.strip():
    print("----- STDERR (last 2000 chars) -----")
    print(proc.stderr[-2000:])
if proc.returncode != 0:
    raise RuntimeError(f"[Error] dicom_to_parquet failed (returncode={proc.returncode}). Check STDERR above.")

# Check _build_summary.json
summary_path = PARQUET_OUT_DIR / "_build_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"[Error] Summary file not found: {summary_path} (extraction did not complete normally)")
build_summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("\n===== _build_summary.json =====")
print(json.dumps(build_summary, ensure_ascii=False, indent=2))
if build_summary.get("files_processed", 0) == 0:
    raise RuntimeError("[Error] 0 files processed. Check DICOM_ROOT path/contents.")
if build_summary.get("rows_total", 0) == 0:
    raise RuntimeError("[Error] 0 tag rows extracted. Check that the DICOM files are valid.")

## ④ Single-Parquet Merge Step
Reads the entire partitioned dataset and merges it into a single `.parquet` file (`SINGLE_PARQUET_PATH`),
also loading it as a DataFrame.

> **Note**: The `dicom_to_parquet` output is (for server storage management) **hive-partitioned** as
> `modality=.../tag=.../part-*.parquet`.
> It must therefore be read with `pyarrow.dataset(..., partitioning="hive")` to restore the partition columns (`modality`, `tag`).
> (Reading individual files with a plain `read_parquet` drops the `modality`/`tag` columns.)
> Also, since `tag` values like `00080016` consist only of digits, they may be misinferred as integers
> and lose their leading zero, so the partition schema is explicitly read as **string**.

In [ ]:
import pyarrow as pa

# Explicitly type the partition columns as string (avoids losing leading zeros)
hive_part = ds.partitioning(
    pa.schema([("modality", pa.string()), ("tag", pa.string())]),
    flavor="hive",
)
dataset = ds.dataset(str(PARQUET_OUT_DIR), format="parquet",
                     partitioning=hive_part, exclude_invalid_files=True)

# Expected columns after merge (11) — 9 file columns + 2 partition columns (modality, tag)
EXPECTED_COLS = ["file_path", "study_uid", "series_uid", "instance_uid",
                 "sop_class_uid", "modality", "tag", "vr", "vm", "path", "value"]

# For large datasets: read in batches and stream-write to a single parquet file
SINGLE_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
writer = None
n_rows = 0
try:
    for batch in dataset.to_batches(batch_size=200_000):
        if batch.num_rows == 0:
            continue
        tbl = pa.Table.from_batches([batch])
        # Reorder columns (only those present)
        cols = [c for c in EXPECTED_COLS if c in tbl.column_names]
        tbl = tbl.select(cols)
        if writer is None:
            writer = pq.ParquetWriter(str(SINGLE_PARQUET_PATH), tbl.schema)
        writer.write_table(tbl)
        n_rows += tbl.num_rows
finally:
    if writer is not None:
        writer.close()

if n_rows == 0:
    raise RuntimeError(f"[Error] No data to merge. Check the parquet files under {PARQUET_OUT_DIR}.")
print(f"Single parquet saved: {SINGLE_PARQUET_PATH}  ({n_rows:,} rows total)")

# Load as DataFrame to inspect
df_raw = pd.read_parquet(SINGLE_PARQUET_PATH)
# Normalize tag/modality to string (tag normalized to 8-digit uppercase hex)
df_raw["tag"] = df_raw["tag"].astype(str).str.upper().str.zfill(8)
df_raw["modality"] = df_raw["modality"].astype(str)
print(f"\ndf_raw shape = {df_raw.shape}")
print("columns:", df_raw.columns.tolist())
print("\nrow count by modality:")
print(df_raw["modality"].value_counts().to_string())
print("\nhead():")
print(df_raw.head().to_string())

## ⑤ df_input Cleaning Step
Converts the `dicom_to_parquet` output schema into the `df_input` schema required by the Evaluator.

### Schema mapping
| df_input column | derivation rule |
|---|---|
| `IOD` | `sop_class_uid` → `pydicom.uid.UID(uid).name` (falls back to the raw UID on failure) |
| `study_instance_uid` | copied from `study_uid` |
| `series_instance_uid` | copied from `series_uid` |
| `Manufacturer` | value of tag `00080070` for the same instance (falls back to the same series, then to "") |
| `ScannerModel` | value of tag `00081090`, resolved the same way |
| `Tag` | `tag` (8-digit uppercase hex) |
| `AttributeName` | `pydicom.datadict.keyword_for_tag(int(tag,16))` (falls back to "") |
| `Value` | value after the preprocessing below |

### Value preprocessing rules
1. Apply `MODALITY_FILTER` (based on modality).
2. **Keep only top-level tags**: keep rows whose `path` matches `"XXXXXXXX[i]/"` (no nesting).
   → Excludes nested rows inside a SQ (`path` contains `[` twice or more) and SQ container rows (value is `"SQ[n]"`).
3. **Merge multi-valued tags (VM>1)**: if the same `(instance_uid, tag)` has multiple value rows,
   merge them into a single row formatted as `"['A', 'B']"` (a list-string the Evaluator parses).

In [ ]:
import re
from pydicom.uid import UID
from pydicom.datadict import keyword_for_tag

MANUFACTURER_TAG = "00080070"   # (0008,0070) Manufacturer
SCANNERMODEL_TAG = "00081090"   # (0008,1090) Manufacturer's Model Name

df = df_raw.copy()

# --- Rule 1: apply MODALITY_FILTER ---
if MODALITY_FILTER:
    before = len(df)
    df = df[df["modality"].isin(MODALITY_FILTER)].copy()
    print(f"[1] Applied MODALITY_FILTER={MODALITY_FILTER}: {before:,} -> {len(df):,} rows")
    if df.empty:
        raise RuntimeError(
            f"[Error] MODALITY_FILTER={MODALITY_FILTER} produced 0 rows.\n"
            f"       Check the actual modality values: {sorted(df_raw['modality'].unique())}")
else:
    print("[1] MODALITY_FILTER is empty -> using all modalities")

# --- Rule 2: keep only top-level (non-nested) tags ---
# top-level scalar path form: 'XXXXXXXX[i]/'  (one bracketed index, no nesting)
#   - SQ container row: path 'XXXXXXXX/' (no brackets) & value 'SQ[n]'  -> excluded
#   - nested row inside SQ: path 'AAAAAAAA[i]/BBBBBBBB[j]/' (2+ bracket pairs) -> excluded
TOP_LEVEL_RE = re.compile(r"^[0-9A-Fa-f]{8}\[\d+\]/$")
top_mask = df["path"].astype(str).str.match(TOP_LEVEL_RE)
# Defensively also exclude SQ container values ('SQ[n]')
sq_mask = df["value"].astype(str).str.match(r"^SQ\[\d+\]$")
top_mask = top_mask & (~sq_mask)
before = len(df)
df = df[top_mask].copy()
print(f"[2] Top-level tags only: {before:,} -> {len(df):,} rows")
if df.empty:
    raise RuntimeError("[Error] 0 rows after top-level tag filter. Check the path format.")

# --- Manufacturer / ScannerModel lookup tables (instance -> series fallback) ---
def _nonempty(s):
    s = "" if s is None else str(s).strip()
    return s if s not in ("", "nan", "[]") else ""

def build_lookup(tag_code):
    sub = df[df["tag"] == tag_code][["instance_uid", "series_uid", "value"]].copy()
    sub["value"] = sub["value"].map(_nonempty)
    sub = sub[sub["value"] != ""]
    inst_map = sub.groupby("instance_uid")["value"].first().to_dict()   # per instance
    ser_map  = sub.groupby("series_uid")["value"].first().to_dict()     # per series (fallback)
    return inst_map, ser_map

manu_inst, manu_ser   = build_lookup(MANUFACTURER_TAG)
model_inst, model_ser = build_lookup(SCANNERMODEL_TAG)

def resolve(row, inst_map, ser_map):
    v = inst_map.get(row["instance_uid"])
    if v:
        return v
    v = ser_map.get(row["series_uid"])
    return v if v else ""

# --- Rule 3: merge multi-valued tags (VM>1) into one row per (instance_uid, tag) ---
df["_idx"] = df["path"].astype(str).str.extract(r"\[(\d+)\]").astype(int)  # multi-value order
grp_keys = ["instance_uid", "study_uid", "series_uid", "sop_class_uid", "tag"]

def merge_values(g):
    vals = g.sort_values("_idx")["value"].astype(str).tolist()
    if len(vals) == 1:
        return vals[0]                 # single value: keep as-is
    return str(vals)                   # multi-value: "['A', 'B']" list string

merged = (df.groupby(grp_keys, sort=False)
            .apply(merge_values, include_groups=False)
            .rename("Value").reset_index())
print(f"[3] Rows after multi-value merge (instance x tag): {len(merged):,}")

# --- Map schema to build df_input ---
def uid_to_iod(uid):
    try:
        name = UID(str(uid)).name
        return name if name else str(uid)
    except Exception:
        return str(uid)

def tag_to_keyword(tag8):
    try:
        kw = keyword_for_tag(int(tag8, 16))
        return kw if kw else ""
    except Exception:
        return ""

df_input = pd.DataFrame({
    "IOD":                 merged["sop_class_uid"].map(uid_to_iod),
    "study_instance_uid":  merged["study_uid"],
    "series_instance_uid": merged["series_uid"],
    "Manufacturer":        merged.apply(lambda r: resolve(r, manu_inst, manu_ser), axis=1),
    "ScannerModel":        merged.apply(lambda r: resolve(r, model_inst, model_ser), axis=1),
    "Tag":                 merged["tag"],
    "AttributeName":       merged["tag"].map(tag_to_keyword),
    "Value":               merged["Value"],
})

# Save (utf-8-sig: Excel Korean-text compatibility)
df_input_path = OUTPUT_DIR / "df_input.csv"
df_input.to_csv(df_input_path, index=False, encoding="utf-8-sig")
print(f"\ndf_input saved: {df_input_path}")
print(f"df_input shape = {df_input.shape}")
print("columns:", df_input.columns.tolist())
print("\nhead():")
print(df_input.head(15).to_string())

## ⑥ Run Quality Evaluation Step
**What**: Feeds the cleaned `df_input` and the standard reference set into the evaluator to compute
**series-level** quality metrics, and saves the final 5 CSV files to `OUTPUT_DIR`.
**Why**: To produce tag_completeness / value_completeness / value_conformance based on the essential tags.

- All CSVs are saved with `encoding="utf-8-sig"` and `group_cols=None` (single group for the whole dataset).

In [ ]:
sys.path.append(str(WORK_DIR / "dicomHeterogeneity" / "DicomStandardEvaluator" / "Evaluator"))
from CxrEssentialTagEvaluator import CxrEssentialTagEvaluator

# Load the standard reference set
df_standard = pd.read_excel(REFERENCE_XLSX)
print(f"Standard reference set loaded: {REFERENCE_XLSX}")
print(f"  standard shape = {df_standard.shape}, number of tags = {df_standard['Tag'].nunique()}")

evaluator = CxrEssentialTagEvaluator(df_input, df_standard)

# 1) Overall completeness/conformance
overall = evaluator.analyze(group_cols=None)
overall_path = OUTPUT_DIR / "cxr_overall_rates.csv"
overall.to_csv(overall_path, index=False, encoding="utf-8-sig")
print(f"\n[1/5] Saved: {overall_path}  (shape={overall.shape})")

# 2~4) 3 conformance sub-reports (summary / partial / none)
conf_prefix = OUTPUT_DIR / "cxr_conformance"
report = evaluator.export_conformance_subreport(str(conf_prefix), group_cols=None)
print(f"[2/5] Saved: {conf_prefix}_summary.csv  (shape={report['summary'].shape})")
print(f"[3/5] Saved: {conf_prefix}_partial.csv  (shape={report['partial'].shape})")
print(f"[4/5] Saved: {conf_prefix}_none.csv     (shape={report['none'].shape})")

# 5) Consolidated non-conforming unique values
unconf_path = OUTPUT_DIR / "cxr_unconformed_values.csv"
df_unconf = evaluator.export_unconformed_values(str(unconf_path), group_cols=None)
print(f"[5/5] Saved: {unconf_path}  (shape={df_unconf.shape})")

## ⑦ Verification Step
**What**: Checks the existence, row count, and column names of the final 5 CSV files in a table,
and summarizes the key metrics (tag_completeness / value_completeness / value_conformance) of `cxr_overall_rates`.
**Why**: To confirm at a glance that the pipeline completed and produced correct output end-to-end.

In [ ]:
EXPECTED_OUTPUTS = [
    "cxr_overall_rates.csv",
    "cxr_conformance_summary.csv",
    "cxr_conformance_partial.csv",
    "cxr_conformance_none.csv",
    "cxr_unconformed_values.csv",
]

print("===== Verifying final 5 CSV files =====")
check_rows = []
all_ok = True
for fn in EXPECTED_OUTPUTS:
    p = OUTPUT_DIR / fn
    if p.exists():
        _df = pd.read_csv(p)
        check_rows.append({"file": fn, "exists": "O", "n_rows": len(_df),
                           "n_cols": _df.shape[1], "columns": ", ".join(map(str, _df.columns))[:80]})
    else:
        all_ok = False
        check_rows.append({"file": fn, "exists": "X", "n_rows": "-",
                           "n_cols": "-", "columns": "(file not found)"})
check_df = pd.DataFrame(check_rows)
print(check_df.to_string(index=False))

if not all_ok:
    raise RuntimeError("[Error] Some final CSV files were not created. Check the 'X' rows in the table above.")
print("\n[OK] All 5 CSV files were created successfully.")

# --- Key metrics summary for cxr_overall_rates ---
print("\n===== cxr_overall_rates key metrics summary =====")
ov = pd.read_csv(OUTPUT_DIR / "cxr_overall_rates.csv")
metric_cols = ["tag_completeness", "value_completeness", "value_conformance"]
total_series = int(ov["total_series"].iloc[0]) if len(ov) else 0
print(f"Total number of series (total_series): {total_series}")
print(f"Number of tags evaluated: {len(ov)}")
print("\n[Metric averages (across all tags)]")
print(ov[metric_cols].mean(numeric_only=True).round(4).to_string())

print("\n[Per-tag metrics (top 20)]")
show_cols = ["Tag", "Attribute Name", "VR", "total_series", "series_with_tag",
             "tag_completeness", "value_completeness", "value_conformance"]
show_cols = [c for c in show_cols if c in ov.columns]
with pd.option_context("display.max_rows", 40, "display.width", 200):
    print(ov[show_cols].head(20).to_string(index=False))